<a href="https://colab.research.google.com/github/vivi0424/HSE-Computational-linguistics/blob/main/deborina_fine_tuning_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [3]:
# Установка библиотек
!pip install transformers datasets evaluate accelerate gradio -q
!pip install huggingface_hub -q

# Для работы с GPU (проверяем наличие)
import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00
GPU доступен: True
Тип GPU: Tesla T4


In [4]:
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# 1. Загрузка датасета
dataset = load_dataset("ag_news")
print(f"Датасет загружен. Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Датасет загружен. Train: 120000, Test: 7600


In [6]:
# 2. Загрузка модели и токенизатора
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4  # 4 класса новостей
).to(device)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
# 3. Токенизация данных
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [8]:
# 4. Настройка обучения
training_args = TrainingArguments(
    output_dir="./results-ag-news",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

In [9]:
# 5. Метрики
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [11]:
# Обучение
train_dataset = tokenized_dataset["train"]
eval_dataset = tokenized_dataset["test"]

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.199447,0.179012,0.942763
2,0.133662,0.184366,0.948026
3,0.085023,0.215418,0.946842


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=22500, training_loss=0.15095956403944227, metrics={'train_runtime': 4207.1243, 'train_samples_per_second': 85.569, 'train_steps_per_second': 5.348, 'total_flos': 1.192249110528e+16, 'train_loss': 0.15095956403944227, 'epoch': 3.0})

In [12]:
# 6. Оценка на тестовой выборке
eval_results = trainer.evaluate()
print(f"Accuracy на тестовой выборке: {eval_results['eval_accuracy']:.4f}")

Accuracy на тестовой выборке: 0.9482


In [13]:
# 7. Сохранение модели
trainer.save_model("./ag_news_model")
tokenizer.save_pretrained("./ag_news_model")
print("Модель сохранена в папку ./ag_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена в папку ./ag_news_model


In [15]:
from transformers import pipeline

In [16]:
# 8. Тестирование на новых новостях
classifier = pipeline(
    "text-classification",
    model="./ag_news_model",
    tokenizer="./ag_news_model"
)

label_names = {
    "LABEL_0": "World",
    "LABEL_1": "Sports",
    "LABEL_2": "Business",
    "LABEL_3": "Sci/Tech"
}

news_texts = [
    "The United Nations held an emergency meeting after rising tensions between several countries in the region.",
    "The football team won the championship after scoring two goals in the final minutes of the match.",
    "A major technology company introduced a new artificial intelligence chip for faster machine learning applications."
]

predictions = classifier(news_texts)

for text, pred in zip(news_texts, predictions):
    print(f"Новость: {text}")
    print(f"Предсказанный класс: {label_names[pred['label']]}")
    print(f"Уверенность модели: {pred['score']:.4f}")
    print()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Новость: The United Nations held an emergency meeting after rising tensions between several countries in the region.
Предсказанный класс: World
Уверенность модели: 0.9935

Новость: The football team won the championship after scoring two goals in the final minutes of the match.
Предсказанный класс: Sports
Уверенность модели: 0.9896

Новость: A major technology company introduced a new artificial intelligence chip for faster machine learning applications.
Предсказанный класс: Sci/Tech
Уверенность модели: 0.9581

